# einops Basics — worked examples (with plain numpy equivalents)

Based on the official tutorial: https://einops.rocks/1-einops-basics/

einops has three core verbs:

| verb | what it does | axes may... |
|------|--------------|-------------|
| `rearrange` | reorder / merge / split axes (no math) | be permuted, **composed** `(a b)`, **decomposed** `(a b)->a b` |
| `reduce`    | rearrange **+** aggregate (`mean/max/min/sum`) | be removed |
| `repeat`    | rearrange **+** broadcast/duplicate | be added |

**Reading a pattern:** `"input axes -> output axes"`. Names are axes; `(a b)` on the
left *splits* a flat axis into `a` then `b`, and on the right *merges* them with `a` as
the slower-varying (outer) index. Lengths that can't be inferred are passed as kwargs
(`b1=2`, `w2=2`, `repeat=3`).

For each example we run einops, show the output, then reproduce it with raw numpy
(`reshape`/`transpose`/`mean`/`tile`/...) and assert the two match exactly — so you can
see precisely what einops expands to.

> The official tutorial uses a bundled `test_images.npy` (6 photos, 96×96, RGB). It
> isn't shipped here, so we **synthesize** an `ims` array of the same shape `(6, 96, 96, 3)`.
> The operations and shapes are identical; only the pixel content differs.

In [1]:
import numpy as np
from einops import rearrange, reduce, repeat
import einops
print("einops", einops.__version__)

# Stand-in for resources/test_images.npy: 6 RGB images, 96x96.
# A smooth ramp so reshapes/transposes are easy to reason about.
ims = np.linspace(0.0, 1.0, 6 * 96 * 96 * 3).reshape(6, 96, 96, 3)
print("ims.shape =", ims.shape)


def check(einops_out, plain_out, tol=1e-9):
    a = np.asarray(einops_out)
    b = np.asarray(plain_out)
    assert a.shape == b.shape, f"shape mismatch: {a.shape} vs {b.shape}"
    assert np.allclose(a, b, atol=tol), "VALUES DIFFER"
    print(f"einops == numpy  ✓   shape = {a.shape}")

einops 0.8.2
ims.shape = (6, 96, 96, 3)


## Part 1 — `rearrange`: reorder, compose, decompose

`rearrange` only moves data around; it never changes values, just their layout.

### 1. Transposition

`"h w c -> w h c"` swaps the first two axes of one image.

In [2]:
e = rearrange(ims[0], "h w c -> w h c")
p = ims[0].transpose(1, 0, 2)
check(e, p)

einops == numpy  ✓   shape = (96, 96, 3)


### 2. Compose batch into height

`"b h w c -> (b h) w c"` stacks the 6 images vertically: `6 * 96 = 576` rows.
Because `b` and `h` are already adjacent and in order, this is a plain `reshape`.

In [3]:
e = rearrange(ims, "b h w c -> (b h) w c")
p = ims.reshape(6 * 96, 96, 3)
print("shape:", e.shape)   # (576, 96, 3)
check(e, p)

shape: (576, 96, 3)
einops == numpy  ✓   shape = (576, 96, 3)


### 3. Compose batch into width

`"b h w c -> h (b w) c"` lays the images side by side. Now `b` must move next to `w`
first, so the numpy form needs a `transpose` *before* the `reshape`.

In [4]:
e = rearrange(ims, "b h w c -> h (b w) c")
p = ims.transpose(1, 0, 2, 3).reshape(96, 6 * 96, 3)   # (h, b, w, c) then merge b,w
print("shape:", e.shape)   # (96, 576, 3)
check(e, p)

shape: (96, 576, 3)
einops == numpy  ✓   shape = (96, 576, 3)


### 4. Decompose then recompose — a 2×3 photo grid

`"(b1 b2) h w c -> (b1 h) (b2 w) c"` with `b1=2` splits the 6 images into a 2×3 grid:
`b1=2` rows go to height, `b2=3` cols go to width → `(192, 288, 3)`.

In [5]:
e = rearrange(ims, "(b1 b2) h w c -> (b1 h) (b2 w) c", b1=2)
p = ims.reshape(2, 3, 96, 96, 3).transpose(0, 2, 1, 3, 4).reshape(2 * 96, 3 * 96, 3)
print("shape:", e.shape)   # (192, 288, 3)
check(e, p)

shape: (192, 288, 3)
einops == numpy  ✓   shape = (192, 288, 3)


### 5. Swap which batch part goes where

`"(b1 b2) h w c -> (b2 h) (b1 w) c"` — same split, but now `b2` drives height and `b1`
drives width → a 3×2 grid `(288, 192, 3)`.

In [6]:
e = rearrange(ims, "(b1 b2) h w c -> (b2 h) (b1 w) c", b1=2)
p = ims.reshape(2, 3, 96, 96, 3).transpose(1, 2, 0, 3, 4).reshape(3 * 96, 2 * 96, 3)
print("shape:", e.shape)   # (288, 192, 3)
check(e, p)

shape: (288, 192, 3)
einops == numpy  ✓   shape = (288, 192, 3)


### 6. Split width, move half to height

`"b h (w w2) c -> (h w2) (b w) c"` with `w2=2` decomposes each width into `w=48, w2=2`
and weaves `w2` into the height axis.

In [7]:
e = rearrange(ims, "b h (w w2) c -> (h w2) (b w) c", w2=2)
p = ims.reshape(6, 96, 48, 2, 3).transpose(1, 3, 0, 2, 4).reshape(96 * 2, 6 * 48, 3)
print("shape:", e.shape)   # (192, 288, 3)
check(e, p)

shape: (192, 288, 3)
einops == numpy  ✓   shape = (192, 288, 3)


### 7. Flatten everything

`"b h w c -> (b h w c)"` merges all axes into one vector. Equivalent to `reshape(-1)`.

In [8]:
e = rearrange(ims, "b h w c -> (b h w c)")
p = ims.reshape(-1)
print("shape:", e.shape)   # (165888,)
check(e, p)

shape: (165888,)
einops == numpy  ✓   shape = (165888,)


### 8. Decomposition only

`"(b1 b2) h w c -> b1 b2 h w c"` with `b1=2` is the inverse of composition: it just
splits the batch into `2 × 3`. A pure `reshape`.

In [9]:
e = rearrange(ims, "(b1 b2) h w c -> b1 b2 h w c", b1=2)
p = ims.reshape(2, 3, 96, 96, 3)
print("shape:", e.shape)   # (2, 3, 96, 96, 3)
check(e, p)

shape: (2, 3, 96, 96, 3)
einops == numpy  ✓   shape = (2, 3, 96, 96, 3)


### 9. Order inside `(...)` matters

`"... -> h (b w) c"` vs `"... -> h (w b) c"` give the **same shape** `(96, 576, 3)` but
**different pixel orderings** — the leftmost name in a group is the slower-varying index.

In [10]:
e1 = rearrange(ims, "b h w c -> h (b w) c")   # blocks of full images
e2 = rearrange(ims, "b h w c -> h (w b) c")   # interleaved columns
print("same shape:", e1.shape == e2.shape, "  identical values:", np.allclose(e1, e2))

check(e1, ims.transpose(1, 0, 2, 3).reshape(96, 6 * 96, 3))   # (h, b, w, c)
check(e2, ims.transpose(1, 2, 0, 3).reshape(96, 96 * 6, 3))   # (h, w, b, c)

same shape: True   identical values: False
einops == numpy  ✓   shape = (96, 576, 3)
einops == numpy  ✓   shape = (96, 576, 3)


## Part 2 — `reduce`: rearrange **and** aggregate

Any axis present on the left but absent on the right is reduced with the named op
(`mean`, `max`, `min`, `sum`, `prod`).

### 10. Average over the batch

`"b h w c -> h w c"` with `"mean"` averages the 6 images into one. = `ims.mean(axis=0)`.

In [11]:
e = reduce(ims, "b h w c -> h w c", "mean")
p = ims.mean(axis=0)
print("shape:", e.shape)   # (96, 96, 3)
check(e, p)

shape: (96, 96, 3)
einops == numpy  ✓   shape = (96, 96, 3)


### 11. Reduce a non-trailing axis (grayscale by averaging channels)

`"b h w c -> b h w"` drops the channel axis. = `ims.mean(axis=3)`.

In [12]:
e = reduce(ims, "b h w c -> b h w", "mean")
p = ims.mean(axis=3)
print("shape:", e.shape)   # (6, 96, 96)
check(e, p)

shape: (6, 96, 96)
einops == numpy  ✓   shape = (6, 96, 96)


### 12. 2×2 max-pooling (decompose-then-reduce)

`"b (h h2) (w w2) c -> h (b w) c"` with `"max", h2=2, w2=2` splits each spatial axis into
blocks of 2, takes the max within each block, and tiles the pooled images side by side.
The numpy form is reshape → `max` over the block axes → transpose/reshape.

In [13]:
e = reduce(ims, "b (h h2) (w w2) c -> h (b w) c", "max", h2=2, w2=2)
p = (ims.reshape(6, 48, 2, 48, 2, 3)
        .max(axis=(2, 4))                 # pool the h2, w2 block axes -> (6,48,48,3)
        .transpose(1, 0, 2, 3)            # (h, b, w, c)
        .reshape(48, 6 * 48, 3))
print("shape:", e.shape)   # (48, 288, 3)
check(e, p)

shape: (48, 288, 3)
einops == numpy  ✓   shape = (48, 288, 3)


### 13. 2×2 mean-pooling, keep batch

`"b (h h2) (w w2) c -> b h w c"` with `"mean"` downsamples each image to 48×48.
= reshape + `mean` over the two block axes.

In [14]:
e = reduce(ims, "b (h h2) (w w2) c -> b h w c", "mean", h2=2, w2=2)
p = ims.reshape(6, 48, 2, 48, 2, 3).mean(axis=(2, 4))
print("shape:", e.shape)   # (6, 48, 48, 3)
check(e, p)

shape: (6, 48, 48, 3)
einops == numpy  ✓   shape = (6, 48, 48, 3)


### 14. Keeping reduced axes as length 1

`"b h w c -> b 1 1 c"` with `"max"` gives the per-image, per-channel maximum but keeps
the spatial axes as size 1 (handy for broadcasting). = `max(axis=(1,2), keepdims=True)`.

In [15]:
e = reduce(ims, "b h w c -> b 1 1 c", "max")
p = ims.max(axis=(1, 2), keepdims=True)
print("shape:", e.shape)   # (6, 1, 1, 3)
check(e, p)

shape: (6, 1, 1, 3)
einops == numpy  ✓   shape = (6, 1, 1, 3)


## Part 3 — Stack & concatenate (list input)

einops treats a **list of arrays** as if a new leading axis already existed. So the same
patterns turn `np.stack` / `np.concatenate` into one readable string.

In [16]:
lst = list(ims)   # a Python list of 6 arrays, each (96, 96, 3)

# Stack along a new leading axis  == np.stack(lst, axis=0)
e = rearrange(lst, "b h w c -> b h w c")
check(e, np.stack(lst, axis=0))

# Concatenate along width  == stack then merge b into w
e = rearrange(lst, "b h w c -> h (b w) c")
check(e, np.stack(lst, 0).transpose(1, 0, 2, 3).reshape(96, 6 * 96, 3))

# Put the stacked axis last  == np.stack(lst, axis=-1)
e = rearrange(lst, "b h w c -> h w c b")
check(e, np.stack(lst, axis=-1))

einops == numpy  ✓   shape = (6, 96, 96, 3)
einops == numpy  ✓   shape = (96, 576, 3)
einops == numpy  ✓   shape = (96, 96, 3, 6)


## Part 4 — Add or remove a length-1 axis

A literal `1` in the pattern inserts (or, on the input side, drops) a unit axis —
einops' version of `np.expand_dims` / `squeeze`.

In [17]:
# Add a trailing unit axis  == ims[..., None]
e = rearrange(ims, "b h w c -> b h w c 1")
check(e, ims[..., None])
print("added:", e.shape)

# Remove it again  == squeeze(-1)
back = rearrange(e, "b h w c 1 -> b h w c")
check(back, e.squeeze(-1))
print("removed:", back.shape)

einops == numpy  ✓   shape = (6, 96, 96, 3, 1)
added: (6, 96, 96, 3, 1)
einops == numpy  ✓   shape = (6, 96, 96, 3)
removed: (6, 96, 96, 3)


## Part 5 — `repeat`: add data by duplication

`repeat` is `rearrange` plus broadcasting: a name on the right that isn't on the left is
a new axis whose length you supply.

### Repeat along an existing axis

`"h w c -> h (repeat w) c"` with `repeat=3` tiles the image 3× horizontally. Because
`repeat` is the outer index of `(repeat w)`, each block is a full copy → `np.tile`.

In [18]:
e = repeat(ims[0], "h w c -> h (repeat w) c", repeat=3)
p = np.tile(ims[0], (1, 3, 1))
print("shape:", e.shape)   # (96, 288, 3)
check(e, p)

shape: (96, 288, 3)
einops == numpy  ✓   shape = (96, 288, 3)


### Repeat into a brand-new axis

`"h w c -> h w c repeat"` with `repeat=3` adds a new trailing axis of 3 identical copies.
= stacking 3 copies along a new last axis.

In [19]:
e = repeat(ims[0], "h w c -> h w c repeat", repeat=3)
p = np.stack([ims[0]] * 3, axis=-1)
print("shape:", e.shape)   # (96, 96, 3, 3)
check(e, p)

shape: (96, 96, 3, 3)
einops == numpy  ✓   shape = (96, 96, 3, 3)


### Repeat along height

`"h w c -> (repeat h) w c"` with `repeat=2` doubles the image vertically. = `np.tile`.

In [20]:
e = repeat(ims[0], "h w c -> (repeat h) w c", repeat=2)
p = np.tile(ims[0], (2, 1, 1))
print("shape:", e.shape)   # (192, 96, 3)
check(e, p)

shape: (192, 96, 3)
einops == numpy  ✓   shape = (192, 96, 3)


## Takeaways

- One readable string replaces chains of `reshape` + `transpose` + `mean`/`max` + `tile`,
  and—unlike bare `reshape`—**names** the axes so the intent (and any bug) is visible.
- `(a b)` composes with `a` as the slower-varying index; the **order inside the group**
  changes the data even when the shape is unchanged (example 9).
- `rearrange` keeps values, `reduce` removes axes (aggregating), `repeat` adds axes
  (duplicating). Lists are treated as a new leading axis, so stacking/concatenation are
  just patterns too.